# Week 4, Notebook 6: Predicting Crime (for Planning)

**Using AI to help Caribbean cities plan safety resources - responsibly.**

*Caribbean AI - Adrian Dunkley*

---

### First, the most important part: ethics

This notebook predicts the **number of reported incidents in a whole city per
month**, so leaders can plan where to send support: more streetlights, youth
programmes, community officers. That is a fair, helpful use of AI.

**What this model must never be used for:**

- Judging or predicting whether an **individual person** will commit a crime.
  That is unjust and the data cannot support it.
- Deciding punishment, bail, or who gets stopped. Predicting a person from a
  group average is discrimination dressed up as maths.
- Sending heavy policing into a neighbourhood in a way that harms the people who
  live there.

A model reflects its data, and crime data reflects **past human decisions**,
including bias. So we predict **city-level totals to guide prevention and
support**, and we stay humble about what the numbers can and cannot say. Keep
this in mind for the whole notebook.

### What you will build
- A neural network that predicts a city's **monthly reported incidents** from
  social and seasonal clues.
- A responsible reading of the results, aimed at **prevention**, not blame.

### The data
Eight Caribbean urban areas (Kingston, Port of Spain, Nassau, Bridgetown,
Georgetown, Castries, San Juan, Santo Domingo), 96 months each.

| Column | What it means |
| --- | --- |
| `population_thousands` | City population (bigger cities see more incidents overall). |
| `unemployment_rate` | Percent out of work. Hardship links to more incidents. |
| `youth_population_pct` | Percent aged 15-24. |
| `police_per_1000` | Community officers per 1,000 people. |
| `is_summer` | 1 in June-August (school out, more people outdoors). |
| `reported_incidents` | What we predict: total reported incidents that month. |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

crime = pd.read_csv("data/caribbean_crime.csv")
print("Rows (city-months):", len(crime))
print("Cities:", list(crime["city"].unique()))
crime.head()

In [ ]:
# Average monthly incidents per city (context, not a ranking of "bad" places -
# bigger cities naturally have larger totals).
avg = crime.groupby("city")["reported_incidents"].mean().sort_values()
plt.figure(figsize=(9, 4.5))
plt.barh(avg.index, avg.values, color="#34495e")
plt.title("Average monthly reported incidents by city\n(larger cities have larger totals)")
plt.xlabel("incidents per month"); plt.tight_layout()
plt.show()

### Step 1: Inputs, split, and scale

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ["population_thousands", "unemployment_rate",
           "youth_population_pct", "police_per_1000", "is_summer"]
X = crime[features].values
y = crime["reported_incidents"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("Train:", len(X_train), " Test:", len(X_test))

### Step 2: A baseline, then the network

Baseline: always predict the **average** number of incidents. Any useful model
must beat that. Then we train the neural network.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
print(f"Baseline MAE (always guess the average): {mean_absolute_error(y_test, baseline_pred):.1f} incidents")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(7)

model = keras.Sequential([
    layers.Input(shape=(len(features),)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(X_train_s, y_train, validation_split=0.15,
                    epochs=200, batch_size=16, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(history.history["mae"], label="training")
plt.plot(history.history["val_mae"], label="validation")
plt.title("Learning curve (MAE)"); plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
nn_pred = model.predict(X_test_s, verbose=0).flatten()
print("--- Test set (unseen city-months) ---")
print(f"Baseline MAE: {mean_absolute_error(y_test, baseline_pred):.1f} incidents")
print(f"Network  MAE: {mean_absolute_error(y_test, nn_pred):.1f} incidents")
print(f"Network  R2:  {r2_score(y_test, nn_pred):.3f}")
print("\nThe network clearly beats 'always guess the average'.")

### Step 3: What drives the prediction?

Permutation importance again: scramble one clue and see how much the error grows.
Read this as *"what should city leaders pay attention to for prevention?"*, not
as blame. Often unemployment and the number of community officers stand out,
which points to **jobs and community presence** as levers for safety.

In [ ]:
rng = np.random.default_rng(0)
base_error = mean_absolute_error(y_test, model.predict(X_test_s, verbose=0).flatten())
imp = []
for j in range(len(features)):
    Xs = X_test_s.copy()
    rng.shuffle(Xs[:, j])
    imp.append(mean_absolute_error(y_test, model.predict(Xs, verbose=0).flatten()) - base_error)

order = np.argsort(imp)
plt.figure(figsize=(8, 4))
plt.barh([features[i] for i in order], [imp[i] for i in order], color="#16a085")
plt.title("What the model leans on most (for prevention planning)")
plt.xlabel("increase in error when the clue is scrambled")
plt.tight_layout()
plt.show()

### Step 4: A prevention "what if"

Suppose a city could lower unemployment or raise community-officer numbers. We
can ask the model what it expects to happen to the monthly total. This is the
kind of question that helps a mayor decide where to invest, always remembering
the model shows **correlations from the past**, not guarantees.

In [ ]:
def predict_city(pop, unemployment, youth, police, summer):
    row = np.array([[pop, unemployment, youth, police, summer]])
    return model.predict(scaler.transform(row), verbose=0)[0, 0]

# A mid-size Caribbean city in summer, comparing two policies.
base = predict_city(300, 16, 25, 3.0, 1)
more_jobs = predict_city(300, 11, 25, 3.0, 1)          # unemployment 16% -> 11%
more_officers = predict_city(300, 16, 25, 4.5, 1)       # 3.0 -> 4.5 per 1000

print(f"Current estimate:            {base:.0f} incidents/month")
print(f"If unemployment 16% -> 11%:  {more_jobs:.0f}  (change {more_jobs-base:+.0f})")
print(f"If officers 3.0 -> 4.5/1000: {more_officers:.0f}  (change {more_officers-base:+.0f})")
print("\nThe model suggests jobs and community presence both help - a case for investment.")

### What you learned

- AI can help cities **plan safety and prevention** by predicting monthly totals
  from social and seasonal clues, and it beats guessing the average.
- **Ethics comes first.** Predict groups and totals for support, never
  individuals for blame. Crime data carries human bias, so stay humble.
- **Feature importance** turns a prediction into **policy insight**: jobs and
  community officers show up as real levers for safety.

### Try it yourself
1. Add a made-up `streetlight_coverage` feature to the data and see if it helps.
2. Run a "what if" for a large city (population 1000) with high unemployment.
3. Discuss with a friend: what is a fair use of this model, and what is not?